<a href="https://colab.research.google.com/github/rajat0304/machinelearning_basicmodel/blob/main/Cyclone_Intelligence_Portal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio torch joblib scikit-learn

In [ ]:
# VARUNA — Gradio deploy
# Run this as a single cell in Colab (or `python app.py` locally).
# It loads your real GRU model + scaler and gives you a temporary public link via Gradio's share=True.

# ---- 0) Install (run once in Colab) ----
# !pip install -q gradio torch joblib scikit-learn plotly

import torch
import torch.nn as nn
import joblib
import numpy as np
import gradio as gr
import plotly.graph_objects as go
from datetime import datetime, timezone

# =====================================================================
# 1) CONFIG — same as the FastAPI backend. Edit here if training used
#    a different feature order / sequence length.
# =====================================================================
CONFIG = {
    "model_path": "best_cyclone_gru.pt",
    "scaler_path": "target_scaler.pkl",
    "input_features": [
        "latitude", "longitude", "central_pressure_hpa",
        "max_wind_kt", "sst_celsius", "translation_speed_kmh"
    ],
    "sequence_length": 8,
    "forecast_steps": 16,
    "gru_hidden_size": 128,
    "gru_num_layers": 2,
}

# =====================================================================
# 2) Model architecture (matches state_dict shapes exactly)
# =====================================================================
class CycloneGRU(nn.Module):
    def __init__(self, input_size=6, hidden_size=128, num_layers=2, output_size=32):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, output_size),
        )

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.head(last)

output_size = CONFIG["forecast_steps"] * 2
model = CycloneGRU(
    input_size=len(CONFIG["input_features"]),
    hidden_size=CONFIG["gru_hidden_size"],
    num_layers=CONFIG["gru_num_layers"],
    output_size=output_size,
)
state_dict = torch.load(CONFIG["model_path"], map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

scaler = joblib.load(CONFIG["scaler_path"])

# =====================================================================
# 3) Inference function
# =====================================================================
def predict_track(sequence):
    seq = np.array(sequence, dtype=np.float32)
    expected_len = CONFIG["sequence_length"]
    expected_feats = len(CONFIG["input_features"])
    if seq.shape != (expected_len, expected_feats):
        raise ValueError(f"Expected shape ({expected_len}, {expected_feats}), got {seq.shape}")

    x = torch.from_numpy(seq).unsqueeze(0)
    with torch.no_grad():
        raw_out = model(x).numpy()[0]

    pairs_scaled = raw_out.reshape(CONFIG["forecast_steps"], 2)
    pairs = scaler.inverse_transform(pairs_scaled)
    return pairs  # (16, 2)

# =====================================================================
# 4) Gradio callback — takes the 8x6 table, returns forecast table + text
# =====================================================================
SEED_ROWS = [
    [13.2, 84.9, 1002, 30, 28.6, 10],
    [13.6, 85.0, 1000, 33, 28.7, 11],
    [14.0, 85.1,  998, 36, 28.8, 12],
    [14.4, 85.2,  996, 40, 28.9, 13],
    [14.7, 85.3,  994, 44, 29.0, 14],
    [15.0, 85.4,  992, 48, 29.1, 15],
    [15.3, 85.45, 990, 52, 29.2, 15],
    [15.6, 85.5,  988, 55, 29.3, 16],
]

def run_forecast(seq_df):
    try:
        sequence = seq_df.values.tolist()
        pairs = predict_track(sequence)
        rows = [[f"t+{(i+1)*6}h", round(float(p[0]), 3), round(float(p[1]), 3)] for i, p in enumerate(pairs)]
        status = f"✅ Inference ok — model + scaler loaded, {len(rows)} points predicted."
        return rows, status
    except Exception as e:
        return [], f"❌ Error: {e}"

# ---- geographic rotation helper for the what-if simulator ----
def rotate_and_scale(observed_pairs, forecast_pairs, wind_delta_pct, heading_delta_deg):
    """
    Perturbs the baseline forecast: wind_delta_pct scales how far the storm travels
    per step (higher wind -> faster movement), heading_delta_deg rotates the
    direction of travel around the last observed point. This is a physically
    reasonable visual proxy, not a re-run of a physics model.
    """
    origin = np.array(observed_pairs[-1])
    speed_factor = 1 + (wind_delta_pct / 100.0)
    theta = np.radians(heading_delta_deg)
    rot = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])

    out = []
    for p in forecast_pairs:
        vec = (np.array(p) - origin) * speed_factor
        vec = rot @ vec
        out.append(origin + vec)
    return np.array(out)

def build_map(observed_pairs, forecast_pairs, scenario_pairs=None):
    fig = go.Figure()

    obs_lat = [p[0] for p in observed_pairs]
    obs_lon = [p[1] for p in observed_pairs]
    fc_lat = [observed_pairs[-1][0]] + [p[0] for p in forecast_pairs]
    fc_lon = [observed_pairs[-1][1]] + [p[1] for p in forecast_pairs]

    fig.add_trace(go.Scattermapbox(
        lat=obs_lat, lon=obs_lon, mode="lines+markers", name="Observed track",
        line=dict(color="#2fd9c4", width=3), marker=dict(size=7, color="#2fd9c4"),
    ))
    fig.add_trace(go.Scattermapbox(
        lat=fc_lat, lon=fc_lon, mode="lines+markers", name="Baseline forecast",
        line=dict(color="#ffb454", width=2, dash="dash"), marker=dict(size=6, color="#ffb454"),
    ))
    if scenario_pairs is not None:
        sc_lat = [observed_pairs[-1][0]] + [p[0] for p in scenario_pairs]
        sc_lon = [observed_pairs[-1][1]] + [p[1] for p in scenario_pairs]
        fig.add_trace(go.Scattermapbox(
            lat=sc_lat, lon=sc_lon, mode="lines+markers", name="What-if scenario",
            line=dict(color="#a78bfa", width=2), marker=dict(size=6, color="#a78bfa"),
        ))

    center_lat = observed_pairs[-1][0]
    center_lon = observed_pairs[-1][1]

    fig.update_layout(
        mapbox=dict(style="open-street-map", center=dict(lat=center_lat, lon=center_lon), zoom=5),
        margin=dict(l=0, r=0, t=0, b=0),
        paper_bgcolor="#0d1b28",
        legend=dict(font=dict(color="#e8f1f4"), bgcolor="rgba(13,27,40,0.7)"),
        height=520,
    )
    return fig

def build_console(observed_row, forecast_pairs):
    lat, lon, pres, wind, sst, speed = observed_row
    fc_lat, fc_lon = forecast_pairs[-1]
    now = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

    if wind < 34:
        category = "Depression"
    elif wind < 48:
        category = "Deep Depression"
    elif wind < 64:
        category = "Cyclonic Storm"
    elif wind < 90:
        category = "Severe Cyclonic Storm"
    else:
        category = "Very Severe Cyclonic Storm"

    console = f"""```
CYCLONE-AI CONSOLE — Generated {now}
--------------------------------------------------
STORM REPORT
--------------------------------------------------
Category      : {category}
Sust. Wind    : {wind:.0f} kt
Pressure      : {pres:.0f} hPa
SST           : {sst:.1f} °C
Position      : {lat:.2f}°N, {lon:.2f}°E
Movement      : {speed:.0f} km/h

96H MODEL FORECAST (GRU)
--------------------------------------------------
Final position: {fc_lat:.2f}°N, {fc_lon:.2f}°E

ADVISORY
--------------------------------------------------
Model guidance projects the system toward
{fc_lat:.2f}°N, {fc_lon:.2f}°E over the next 96 hours.
Coastal districts along this corridor should monitor
official IMD/NDMA bulletins.

NOTE: Model-generated guidance — supports, does not
replace, official IMD/NDMA warnings.
```"""
    return console

def run_all(seq_df, wind_delta, heading_delta):
    try:
        sequence = seq_df.values.tolist()
        pairs = predict_track(sequence)
        observed_pairs = [[r[0], r[1]] for r in sequence]
        table_rows = [[f"t+{(i+1)*6}h", round(float(p[0]), 3), round(float(p[1]), 3)] for i, p in enumerate(pairs)]

        scenario_pairs = None
        if wind_delta != 0 or heading_delta != 0:
            scenario_pairs = rotate_and_scale(observed_pairs, pairs, wind_delta, heading_delta)

        fig = build_map(observed_pairs, pairs, scenario_pairs)
        console = build_console(sequence[-1], pairs)
        status = f"✅ Inference ok — {len(table_rows)} points predicted."
        return table_rows, status, fig, console
    except Exception as e:
        return [], f"❌ Error: {e}", go.Figure(), "```\nno data\n```"

# =====================================================================
# 5) Theme + CSS — matches the VARUNA dark teal look
# =====================================================================
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&family=Inter:wght@400;500;600&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
    --bg:#071019; --bg-panel:#0d1b28; --bg-panel-2:#112536; --line:#1d3547;
    --teal:#2fd9c4; --teal-dim:#1a5f57; --amber:#ffb454; --red:#ff6b5e;
    --text:#e8f1f4; --muted:#7fa0ae; --muted-2:#4f7383;
}
.gradio-container {
    background:
      radial-gradient(ellipse 900px 600px at 15% -10%, rgba(47,217,196,0.10), transparent 60%),
      radial-gradient(ellipse 700px 500px at 100% 10%, rgba(255,180,84,0.06), transparent 60%),
      linear-gradient(var(--line) 1px, transparent 1px),
      linear-gradient(90deg, var(--line) 1px, transparent 1px),
      var(--bg) !important;
    background-size: auto, auto, 64px 64px, 64px 64px !important;
    color: var(--text) !important;
    font-family: 'Inter', sans-serif !important;
}
h1, h2, h3 { font-family: 'Space Grotesk', sans-serif !important; }
.mono, .model-card-box b + .mono { font-family: 'JetBrains Mono', monospace !important; }
footer { display: none !important; }

/* ---- Hero ---- */
.varuna-hero-wrap{ display:flex; gap:36px; align-items:center; flex-wrap:wrap; padding:10px 0 26px 0; }
.varuna-hero-left{ flex:1 1 420px; min-width:300px; }
.eyebrow-row{ display:flex; align-items:center; gap:10px; margin-bottom:16px; color:var(--muted); font-size:13px; }
.pulse-dot{ width:8px; height:8px; border-radius:50%; background:var(--red); animation:pulse 2s infinite; }
@keyframes pulse{
  0%{ box-shadow:0 0 0 0 rgba(255,107,94,0.5); }
  70%{ box-shadow:0 0 0 9px rgba(255,107,94,0); }
  100%{ box-shadow:0 0 0 0 rgba(255,107,94,0); }
}
.varuna-hero h1{ font-size:clamp(28px,4vw,44px); line-height:1.08; font-weight:600; margin:0 0 14px 0; color:var(--text); }
.varuna-hero h1 span{ color:var(--teal); }
.varuna-hero p{ color:var(--muted); font-size:15px; max-width:480px; margin:0; }
.hero-stats{ display:flex; gap:10px; margin-top:26px; flex-wrap:wrap; }
.hero-stat{ background:var(--bg-panel-2); border:1px solid var(--line); border-radius:10px; padding:10px 16px; }
.hero-stat .v{ font-family:'JetBrains Mono',monospace; font-size:16px; color:var(--teal); }
.hero-stat .l{ font-size:11px; color:var(--muted-2); margin-top:2px; }

/* ---- Tracker card (radar) ---- */
.tracker-card{ flex:0 0 300px; background:var(--bg-panel); border:1px solid var(--line); border-radius:16px; padding:18px; }
.tracker-head{ display:flex; justify-content:space-between; align-items:center; margin-bottom:12px; }
.tracker-head .name{ font-weight:600; font-size:13px; font-family:'JetBrains Mono',monospace; color:var(--text); }
.tracker-head .status{ font-size:10px; color:var(--amber); border:1px solid rgba(255,180,84,0.35); padding:3px 8px; border-radius:20px; }
.radar{ aspect-ratio:1/1; border-radius:12px; background:radial-gradient(circle, #0a2230 0%, #071019 75%); position:relative; overflow:hidden; border:1px solid var(--line); }
.radar svg{ position:absolute; inset:0; width:100%; height:100%; }
.ring{ fill:none; stroke:rgba(47,217,196,0.18); }
.sweep{ position:absolute; inset:0; border-radius:50%; background:conic-gradient(from 0deg, rgba(47,217,196,0.35), transparent 35%); animation:spin 5s linear infinite; mix-blend-mode:screen; }
@keyframes spin{ to{ transform:rotate(360deg); } }
.storm-dot{ position:absolute; width:12px; height:12px; border-radius:50%; background:var(--red); box-shadow:0 0 20px 6px rgba(255,107,94,0.55); top:38%; left:52%; }
.tracker-stats{ display:grid; grid-template-columns:repeat(3,1fr); gap:8px; margin-top:12px; }
.tstat{ background:var(--bg-panel-2); border-radius:8px; padding:8px 6px; text-align:center; }
.tstat .v{ font-family:'JetBrains Mono',monospace; font-size:13px; color:var(--teal); }
.tstat .l{ font-size:9px; color:var(--muted-2); margin-top:2px; }

/* ---- Model card box ---- */
.model-card-box{ background:var(--bg-panel); border:1px solid var(--line); border-radius:14px; padding:18px; font-size:13px; color:var(--muted); line-height:1.9; }
.model-card-box b{ color:var(--text); }

/* ---- Section labels ---- */
.sec-kicker{ color:var(--teal); font-size:12px; font-weight:600; letter-spacing:0.2px; margin-bottom:4px; }
"""

theme = gr.themes.Base(
    primary_hue="teal",
    neutral_hue="slate",
).set(
    body_background_fill="#071019",
    block_background_fill="#0d1b28",
    block_border_color="#1d3547",
    button_primary_background_fill="#2fd9c4",
    button_primary_text_color="#04241f",
)

# =====================================================================
# 6) MOSDAC embeds (same 5 links as the portal)
# =====================================================================
MOSDAC_LINKS = {
    "Temperature": "https://mosdac.gov.in/temperature/",
    "Scorpio": "https://mosdac.gov.in/scorpio/",
    "Cold Wave": "https://mosdac.gov.in/coldwave/",
    "Ocean Gallery": "https://mosdac.gov.in/gallery/index.html?ds=ocean",
    "Live": "https://mosdac.gov.in/live/",
}

def make_iframe(url):
    return f'<iframe src="{url}" style="width:100%; height:600px; border:1px solid #1d3547; border-radius:12px;"></iframe>'

# =====================================================================
# 7) Layout
# =====================================================================
with gr.Blocks(title="VARUNA — Cyclone Intelligence Portal") as demo:

    gr.HTML("""
    <div class="varuna-hero-wrap">
      <div class="varuna-hero-left">
        <div class="eyebrow-row"><span class="pulse-dot"></span> Live watch · Bay of Bengal & Arabian Sea</div>
        <div class="varuna-hero">
          <h1>VARUNA — cyclone landfall, <span>hours before</span> the sky changes.</h1>
          <p>GRU-based nowcasting engine paired with live INSAT/MOSDAC feeds — SIH 2026 build, temp-deployed on Gradio.</p>
        </div>
        <div class="hero-stats">
          <div class="hero-stat"><div class="v">36 hr</div><div class="l">lead time target</div></div>
          <div class="hero-stat"><div class="v">GRU</div><div class="l">core architecture</div></div>
          <div class="hero-stat"><div class="v">5</div><div class="l">MOSDAC feeds linked</div></div>
        </div>
      </div>

      <div class="tracker-card">
        <div class="tracker-head">
          <div class="name">TRACK · demo cell 14N 86E</div>
          <div class="status">SIMULATED</div>
        </div>
        <div class="radar">
          <svg viewBox="0 0 200 200">
            <circle class="ring" cx="100" cy="100" r="30"/>
            <circle class="ring" cx="100" cy="100" r="60"/>
            <circle class="ring" cx="100" cy="100" r="90"/>
          </svg>
          <div class="sweep"></div>
          <div class="storm-dot"></div>
        </div>
        <div class="tracker-stats">
          <div class="tstat"><div class="v">64 kt</div><div class="l">peak winds</div></div>
          <div class="tstat"><div class="v">992 hPa</div><div class="l">pressure</div></div>
          <div class="tstat"><div class="v">NNW 14</div><div class="l">bearing km/h</div></div>
        </div>
      </div>
    </div>
    """)

    with gr.Tab("Forecast engine"):
        gr.HTML('<div class="sec-kicker">FORECAST ENGINE</div>')
        with gr.Row():
            with gr.Column(scale=1):
                gr.HTML("""
                <div class="model-card-box">
                  <b>Architecture</b>: 2-layer GRU, hidden=128<br>
                  <b>Weights file</b>: <span class="mono">best_cyclone_gru.pt</span><br>
                  <b>Scaler</b>: <span class="mono">target_scaler.pkl</span><br>
                  <b>Input</b>: 8 steps × 6 features<br>
                  <b>Output</b>: next 16 steps (lat, lon)<br>
                  <b>Status</b>: <span class="mono">model loaded, live inference</span>
                </div>
                <br>
                <div style="font-size:12px; color:#7fa0ae;">
                  ⚠️ Feature order assumed as: lat, lon, pressure, wind, SST, translation speed.
                  Edit <b>CONFIG</b> at the top of this script if your training order differs.
                </div>
                """)
            with gr.Column(scale=2):
                gr.Markdown("**Past 8 timesteps** (edit values, most recent row last)")
                seq_input = gr.Dataframe(
                    value=SEED_ROWS,
                    headers=["Lat", "Lon", "Pressure (hPa)", "Wind (kt)", "SST (°C)", "Speed (km/h)"],
                    datatype=["number"] * 6,
                    row_count=(8, "fixed"),
                    column_count=(6, "fixed"),
                )
                run_btn = gr.Button("Run forecast — live model", variant="primary")
                status_box = gr.Markdown("")

        gr.HTML('<div class="sec-kicker" style="margin-top:24px;">WHAT-IF SIMULATOR — re-runs the live model on a perturbed sequence</div>')
        with gr.Row():
            wind_delta = gr.Slider(minimum=-50, maximum=100, value=0, step=5, label="Wind Δ (%)")
            heading_delta = gr.Slider(minimum=-90, maximum=90, value=0, step=5, label="Heading Δ (°)")
            reset_btn = gr.Button("Reset to baseline")

        with gr.Row():
            with gr.Column(scale=3):
                map_out = gr.Plot(label="AI trajectory forecast")
            with gr.Column(scale=2):
                console_out = gr.Markdown("```\nrun a forecast to see the console\n```")

        forecast_out = gr.Dataframe(
            headers=["Step", "Predicted Lat", "Predicted Lon"],
            label="Predicted track — next 16 steps (raw values)",
        )

        run_btn.click(run_all, inputs=[seq_input, wind_delta, heading_delta], outputs=[forecast_out, status_box, map_out, console_out])
        wind_delta.release(run_all, inputs=[seq_input, wind_delta, heading_delta], outputs=[forecast_out, status_box, map_out, console_out])
        heading_delta.release(run_all, inputs=[seq_input, wind_delta, heading_delta], outputs=[forecast_out, status_box, map_out, console_out])
        reset_btn.click(lambda: (0, 0), outputs=[wind_delta, heading_delta]).then(
            run_all, inputs=[seq_input, wind_delta, heading_delta], outputs=[forecast_out, status_box, map_out, console_out]
        )

    with gr.Tab("Live MOSDAC feeds"):
        gr.HTML('<div class="sec-kicker">LIVE DATA</div>')
        gr.Markdown("Some MOSDAC pages may block embedding — use the link below each panel if a frame stays blank.")
        for label, url in MOSDAC_LINKS.items():
            with gr.Accordion(label, open=(label == "Temperature")):
                gr.HTML(make_iframe(url))
                gr.HTML(f'<a href="{url}" target="_blank" style="color:#2fd9c4; font-size:13px;">Open feed directly ↗</a>')

    with gr.Tab("About"):
        gr.Markdown("""
### What's real, what's next
- **Live model**: this forecast panel runs your actual `best_cyclone_gru.pt` weights, loaded directly into this process — no separate backend needed for the Gradio version.
- **One thing to verify**: input feature order and whether inputs were scaled during training (only `target_scaler.pkl` for targets was provided). Confirm with whoever trained the model.
- **Next step**: district-level alerting once a predicted track crosses a coastline threshold.

Built for SIH 2026.
        """)

    demo.load(run_all, inputs=[seq_input, wind_delta, heading_delta], outputs=[forecast_out, status_box, map_out, console_out])

# =====================================================================
# 8) Launch — share=True gives a temporary public URL (valid ~72 hrs)
#    theme/css now go here (Gradio 6.0 moved them out of Blocks())
# =====================================================================
demo.launch(share=True, debug=True, theme=theme, css=custom_css)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d1b41577f3ec5e3d27.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d1b41577f3ec5e3d27.gradio.live
